# Consolidated 1000 epigenomes

"Universal annotation of the human genome through integration of over a thousand epigenomic datasets"
https://link.springer.com/article/10.1186/s13059-021-02572-z

See https://egg2.wustl.edu/roadmap/data/byFileType/metadata/EID_metadata.tab

The segmentations are produced by `process_epi1000.sh`.

Five sections, each of them **Compute** (tables and caches under `out/`), **Plots**
(figures written to `out/*.png`, none rendered) and **Show** (the figures rendered
off disk):

1. **De-novo methods** - ChromHMM on the default binarization and the KMeans
   segmentations of the three peak callers, each individually and jointly.
2. **Reference 18-state core K27ac model** - the published Roadmap segmentation.
3. **Reference 15-state core model** - published per epigenome vs the joint model
   reordered onto the same states.
4. **Cell type differences** - how many epigenomes call a state in the same place.
5. **Replicates consistency** - agreement within the groups of `epi1000_replicates.yaml`.

## Setup

In [ ]:
import contextlib
import glob
import importlib
import io
import multiprocessing as mp
import os
import sys
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from tqdm.auto import tqdm

# Fixed on the first run: the next cell chdirs into the working directory, so a
# re-run of this one must not resolve the project root against it again.
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = os.getcwd()

# The analysis methods are imported directly, no CLI / subprocess.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts", "analysis"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts", "rules"))

import analyze
import analyze_peaks
import compare
import match
import peaks_segmentation
import summary_plots
import utils
import display as _display_helpers
from utils import (
    CHROMHMM_DEFAULT, COMPOSITION_DISPLAY, COSINE, COSINE_DISPLAY,
    EMISSION_DISPLAY, FULL, FULL_DISPLAY, HOMER, JACCARD, JACCARD_DISPLAY,
    JOINT_CHROMHMM, JOINT_KMEANS_HOMER, JOINT_KMEANS_MACS2, JOINT_KMEANS_OMNI,
    KAPPA, KAPPA_DISPLAY, KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI, MACS2, NOQH,
    NOQH_DISPLAY, NOQH_STATES, OMNI
)

# Re-import the analysis modules so edits to scripts/*/*.py are picked up when
# this cell is re-run, without a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, match, peaks_segmentation,
           summary_plots, utils, _display_helpers):
    importlib.reload(_m)

# Bound after the reload above so edits to display.py are picked up too.
# Imported as functions, not as the module: the bare name `display` belongs to
# IPython's own display().
from display import header, show, show_all

In [ ]:
# Configuration is shared with the ENCODE analysis; only the working directory
# differs, and every out/... path below is relative to it.
with open(os.path.join(PROJECT_ROOT, "config_encode.yaml")) as f:
    encode_config = yaml.safe_load(f)

EPI_1000_PATH = os.path.expanduser("~/data/2026_segmentations/epi1000")
os.chdir(EPI_1000_PATH)
os.makedirs("out", exist_ok=True)

P = encode_config["params"]
MARKS = ["H3K36me3", "H3K9me3", "H3K4me1", "H3K27ac", "H3K27me3", "H3K4me3"]
CALLERS = [HOMER, MACS2, OMNI]
CHROMHMM_BIN = P["chromhmm_bin"]

# process_epi1000.sh binarized every peak caller at one bin size and segmented
# all of them from that, so a KMeans segmentation here is at 100 bp whatever
# per-caller bin config_encode.yaml carries for the Snakemake pipeline (where
# rules/kmeans.smk does give each caller its own bin). Reading it from the
# config instead put the homer segmentations at 200 bp, twice their real
# resolution, which halved every bin count derived from them.
KMEANS_BIN = 100

# Pairs per pairwise comparison: every pair of a hundred epigenomes is tens of
# thousands of them, read off a reproducible sample instead.
PAIR_LIMIT = 1000

# The metrics and comparison domains every agreement below is measured in, as
# (display name, on-disk key) - the key names the cache column and the file.
METRICS = [(JACCARD_DISPLAY, JACCARD), (KAPPA_DISPLAY, KAPPA), (COSINE_DISPLAY, COSINE)]
DOMAINS = [(FULL_DISPLAY, FULL), (NOQH_DISPLAY, NOQH)]

# The composition axis is broken between these two: the Quiescent state covers
# more than half of the genome and flattens every other state on a shared axis.
BREAK_LOW, BREAK_HIGH = 0.20, 0.40
# Marker style of a bar carrying hundreds of points (state x method x dataset).
CROWDED_POINTS = {"size": 1.5, "alpha": 0.4, "jitter": 0.2}
PAIR_POINTS = {"size": 1, "alpha": 0.25}


def seg_bin(path):
    """Bin size a segmentation of this dataset is written at."""
    if any(f"/{caller}/" in path for caller in CALLERS):
        return KMEANS_BIN
    return CHROMHMM_BIN


def mixed_background(*lengths):
    """The background states of segmentations that mix naming conventions.

    The de-novo models name the background "Quies" / "Het" and the published
    ones "15_Quies" / "9_Het", so a pair of them is matched by substring rather
    than by the exact state names of NOQH_STATES.
    """
    return {s for lengths_of_side in lengths for s in lengths_of_side
            if any(bg in s for bg in NOQH_STATES)}


def pair_agreement(path1, path2, background=None):
    """Agreement of two segmentations of the genome, per comparison domain."""
    s1, s2 = match.load_bed(path1), match.load_bed(path2)
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    return match.agreement_by_mode(
        match.pair_overlap(s1, s2), l1, l2,
        background=mixed_background(l1, l2) if background is None else background)


def load_side(cache, path):
    """(segments, state lengths) of *path*, reusing the last loaded pair side.

    The pair lists are sorted, so the left side of a pair stays the same over a
    run of pairs and is loaded once for all of them; *cache* is a dict the
    caller keeps between calls, and holds one segmentation at a time.
    """
    if path not in cache:
        cache.clear()
        segs = match.load_bed(path)
        cache[path] = (segs, match.state_lengths(segs))
    return cache[path]


def per_state_rows(overlaps, states, method=None):
    """Per-state agreement of every pair of *overlaps*, in long form.

    One row per pair, state and metric: the diagonal of the state-by-state
    agreement matrix, which is how much of a state the two sides place in the
    same bp. Restricted to *states*, so what a model never calls scores
    nothing rather than a free 1.0.
    """
    rows = []
    for overlap in overlaps:
        for metric, metric_key in METRICS:
            for state, value in match.per_state_diagonal(overlap, states, metric_key).items():
                rows.append({"Method": method, "State": state,
                             "Metric": metric, "Value": value})
    return pd.DataFrame(rows)


def methods_current(df, order):
    """True when the Method column of *df* is keyed the way *order* is.

    A cache written before the methods were renamed from display names
    ("Default ChromHMM") to keys ("chromhmm_default") has exactly the right
    shape, so a row count alone accepts it - and then pd.Categorical() turns
    every Method into NaN and every per-method plot silently empties.
    """
    return set(df["Method"].dropna().unique()) <= set(order)


print(f"Working dir : {EPI_1000_PATH}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Callers     : {CALLERS}")

## 1. De-novo methods

ChromHMM on the default binarization and the KMeans segmentations of the three
peak callers, each of them called per epigenome and jointly over all of them,
with the states matched onto the ENCODE 15-state reference.

### Compute

In [ ]:
# One segmentation per epigenome folder and method, relative to the folder.
METHOD_PATTERNS = {
    CHROMHMM_DEFAULT:   "{f}_chromhmm/{f}_15_dense_matched.bed",
    KMEANS_HOMER:       "homer/{f}_homer_kmeans_states_matched.bed",
    KMEANS_MACS2:       "macs2/{f}_macs2_kmeans_states_matched.bed",
    KMEANS_OMNI:        "omni/{f}_omni_kmeans_states_matched.bed",
    JOINT_KMEANS_HOMER: "../joint_kmeans/homer/{f}_kmeans_joint_states_matched.bed",
    JOINT_KMEANS_MACS2: "../joint_kmeans/macs2/{f}_kmeans_joint_states_matched.bed",
    JOINT_KMEANS_OMNI:  "../joint_kmeans/omni/{f}_kmeans_joint_states_matched.bed",
    JOINT_CHROMHMM:     "../joint_chromhmm/all_epigenomes_15_result/{f}_15_dense.bed",
}

EPI_FOLDERS = sorted(d for d in os.listdir(EPI_1000_PATH)
                     if d.startswith("E") and os.path.isdir(os.path.join(EPI_1000_PATH, d)))

# State colours of the ENCODE reference the states above are matched onto.
ref_colors = match.state_colors(match.load_bed(
    os.path.expanduser(encode_config["workdir"] + "/monocytes/ENCFF227EMB_chromhmm.bed")))

found = [(folder, method, path)
         for folder in EPI_FOLDERS
         for method, pattern in METHOD_PATTERNS.items()
         if os.path.exists(path := os.path.join(EPI_1000_PATH, folder,
                                                pattern.format(f=folder)))]


def state_names_matched(path):
    """True when every segment of *path* carries a matched state name."""
    try:
        return all(row[3] and row[3] != "." for row in match.load_bed(path))
    except Exception:
        return False


print(f"Checking {len(found)} existing segmentations...")
with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
    matched = list(tqdm(executor.map(state_names_matched, [p for _, _, p in found]),
                        total=len(found)))

# A segmentation whose states were never matched has no comparable state names.
dropped = [t for t, ok in zip(found, matched) if not ok]
if dropped:
    print(f"Filtering out {len(dropped)} segmentations with missing state names:")
    for folder, method, _ in dropped:
        print(f"  - {folder} {method}")

valid_tasks = [(folder, method, path, seg_bin(path))
               for (folder, method, path), ok in zip(found, matched) if ok]
seg_paths = [path for _, _, path, _ in valid_tasks]
seg_bins = [bin_size for _, _, _, bin_size in valid_tasks]

# Only the methods that have a segmentation, in the order of METHOD_PATTERNS.
method_order = [m for m in METHOD_PATTERNS if any(t[1] == m for t in valid_tasks)]
method_palette = {m: utils.method_color(m) for m in method_order}
method_idxs = {m: [i for i, t in enumerate(valid_tasks) if t[1] == m] for m in method_order}
seg_by_key = {(folder, method): path for folder, method, path, _ in valid_tasks}
print(f"{len(valid_tasks)} segmentations over {len(method_order)} methods: "
      + ", ".join(f"{m} ({len(idxs)})" for m, idxs in method_idxs.items()))

In [ ]:
# Per-segmentation tables: state and segment counts, transition entropy and the
# state composition. Each of them is cached, so a re-run reads them back.
def _transitions(i):
    return analyze.build_transition_matrix(match.load_bed(seg_paths[i]), seg_bins[i])


def _transitions_noqh(i):
    return analyze.build_transition_matrix(match.load_bed(seg_paths[i]), seg_bins[i],
                                           exclude_states=NOQH_STATES)


def _segment_stats(i):
    return compare.compute_segment_stats(match.load_bed(seg_paths[i]))


def _composition(i):
    folder, method, path, _ = valid_tasks[i]
    return [{"Dataset": folder, "Method": method, **state}
            for state in analyze.state_composition(match.load_bed(path))]


def _parallel(worker, label):
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
        return list(tqdm(executor.map(worker, range(len(valid_tasks))),
                         total=len(valid_tasks), desc=label))


# A cache from a different set of segmentations is not reusable at all. v2: the
# segmentations are binned by seg_bin(), whose homer bin size was corrected.
all_tm, all_tm_noqh = utils.cached_pickle(
    "out/all_tm.pkl",
    lambda: (_parallel(_transitions, "Transitions"),
             _parallel(_transitions_noqh, "Transitions NOQH")),
    label="transition matrices",
    valid=lambda cached: len(cached[0]) == len(valid_tasks))

all_stats = utils.cached_pickle(
    "out/all_stats.pkl", lambda: _parallel(_segment_stats, "Segment stats"),
    label="segmentation statistics",
    valid=lambda cached: len(cached) == len(valid_tasks))


def compute_results():
    rows = []
    for i, (folder, method, _, _) in enumerate(valid_tasks):
        entropy = analyze.transition_entropy(*all_tm[i])[0]
        states_noqh = all_tm_noqh[i]
        rows.append({
            "Dataset": folder,
            "Method": method,
            "N_States": all_stats[i].get("n_states", 0),
            "N_Segments": all_stats[i].get("n_segments", 0),
            "Entropy": entropy,
            "Entropy_NOQH": analyze.transition_entropy(*states_noqh)[0] if states_noqh[0] else 0,
        })
    return pd.DataFrame(rows)


df_results = utils.cached_csv(
    "out/df_results.csv", compute_results, label="results",
    valid=lambda df: len(df) == len(valid_tasks) and methods_current(df, method_order))
df_results["Method"] = pd.Categorical(df_results["Method"], categories=method_order, ordered=True)

df_comp = utils.cached_csv(
    "out/df_comp.csv",
    lambda: pd.DataFrame([row for rows in _parallel(_composition, "Composition") for row in rows]),
    label="state composition",
    valid=lambda df: len(df.groupby(["Dataset", "Method"])) == len(valid_tasks)
                     and "MeanLength" in df.columns and methods_current(df, method_order))
df_comp["Method"] = pd.Categorical(df_comp["Method"], categories=method_order, ordered=True)

# summary_plots.sort_states puts numbered names ("1_TssA", "10_TssB") in state
# order; the colours are the ones the segmentations themselves carry, sampled
# from a few per method, with the reference and the canonical ones behind them.
states_order = summary_plots.sort_states(df_comp["State"].unique())
seg_colors = {}
for idxs in method_idxs.values():
    for i in idxs[:3]:
        for state, color in match.state_colors(match.load_bed(seg_paths[i])).items():
            seg_colors.setdefault(state, color)
state_colors = summary_plots.state_palette(states_order, seg_colors, ref_colors)

In [ ]:
# Pairwise consistency: how much two epigenomes segmented by the same method
# agree, over a sample of at most PAIR_LIMIT pairs. Two tables come out of the
# same overlaps - the pooled metrics per method (df_pw) and the agreement of
# every state with itself (df_pw_state).
pair_indices = {}   # segmentation index of a pair side, shared with the workers


def _pair_state_lengths(i):
    return match.state_lengths(match.load_bed(seg_paths[i]))


def _pair_overlap(pair):
    _, i, j = pair
    return match.pair_overlap(None, match.load_bed(seg_paths[j]),
                              ref_index=pair_indices[i])


def method_pairwise(method):
    """{overlaps, lengths, df} of the sampled pairs of one method."""
    idxs = method_idxs[method]
    pairs = utils.sample_pairs([(method, i, j) for i, j in combinations(idxs, 2)], PAIR_LIMIT)
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
        lengths = dict(zip(idxs, tqdm(executor.map(_pair_state_lengths, idxs, chunksize=10),
                                      total=len(idxs), desc="Lengths", leave=False)))
    # The left side of every pair is indexed once, rather than per pair.
    pair_indices.clear()
    pair_indices.update({i: match.build_index(match.load_bed(seg_paths[i]))
                         for i in tqdm(sorted({i for _, i, _ in pairs}),
                                       desc="Indexing", leave=False)})
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
        overlaps = list(tqdm(executor.map(_pair_overlap, pairs, chunksize=100),
                             total=len(pairs), desc=f"Overlaps {method}"))
    pair_indices.clear()

    rows = []
    for (_, i, j), overlap in zip(pairs, overlaps):
        metrics = match.agreement_by_mode(overlap, lengths[i], lengths[j],
                                          background=NOQH_STATES)
        for mode, m in metrics.items():
            rows.append({"Method": method, "Mode": utils.domain_display(mode),
                         JACCARD_DISPLAY: m[JACCARD], KAPPA_DISPLAY: m[KAPPA],
                         COSINE_DISPLAY: m[COSINE]})
    return {"overlaps": overlaps, "lengths": lengths, "df": pd.DataFrame(rows)}


pw_frames, pw_state_frames = [], []
for method in method_order:
    if len(method_idxs[method]) < 2:
        print(f"No pairs for {method}, skipping")
        continue
    # One cache per method, which summary.ipynb reads the cross-sample evidence
    # of that method out of - it collects the "df" of every out/pw_cache_*.pkl.
    cache = utils.cached_pickle(
        f"out/pw_cache_{method}.pkl", lambda m=method: method_pairwise(m),
        label=f"pairwise consistency for {method}",
        valid=lambda c: {"df", "overlaps", "lengths"} <= set(c))
    pw_frames.append(cache["df"])
    # The states this method realizes, in state order; a state it never calls
    # is left out of its per-state agreement rather than scored zero.
    called = (cache["lengths"].get(method_idxs[method][0])
              or {s for l in cache["lengths"].values() for s in l})
    pw_state_frames.append(per_state_rows(cache["overlaps"],
                                          [s for s in states_order if s in called],
                                          method=method))
    del cache   # the overlaps of one method are tens of MB

def _concat(frames, columns):
    """The frames of every method as one, or an empty frame of *columns*."""
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    df["Method"] = pd.Categorical(df["Method"], categories=method_order, ordered=True)
    return df


df_pw = _concat(pw_frames, ["Method", "Mode"] + [m for m, _ in METRICS])
df_pw = df_pw.sort_values(["Method", "Mode"]).reset_index(drop=True)
df_pw_state = _concat(pw_state_frames, ["Method", "State", "Metric", "Value"])
print(f"{len(df_pw)} pooled and {len(df_pw_state)} per-state pairwise rows")

In [ ]:
# Peak statistics of the callers the KMeans segmentations are built from.
def _peak_stats(folder):
    folder_path = os.path.join(EPI_1000_PATH, folder)
    outdir = os.path.join(folder_path, "peaks")
    analyze_peaks.run_analyze_peaks(folder_path, folder, list(MARKS), outdir,
                                    omni_bin=P["omni_bin"], chromhmm_bin=CHROMHMM_BIN)
    tsv = os.path.join(outdir, "peak_stats.tsv")
    if not os.path.exists(tsv):
        return None
    return pd.read_csv(tsv, sep="\t").assign(dataset=folder)


def compute_peak_stats():
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
        frames = list(tqdm(executor.map(_peak_stats, EPI_FOLDERS),
                           total=len(EPI_FOLDERS), desc="Peak stats"))
    return pd.concat([f for f in frames if f is not None], ignore_index=True)


df_peaks = utils.cached_csv("out/df_peaks.csv", compute_peak_stats, label="peak statistics")

### Plots

In [ ]:
# Peak counts and mean peak length; the 5-95% outliers are logged, not dropped.
summary_plots.log_peak_outliers(df_peaks)
summary_plots._plot_peak_count(df_peaks, EPI_1000_PATH, "out/n_peaks.png")
summary_plots._plot_peak_length(df_peaks, EPI_1000_PATH, "out/peak_length.png")

# Per-method summaries. Segments in thousands, for consistency with the other
# analysis notebooks.
df_results["N_Segments_K"] = df_results["N_Segments"] / 1000.0
for column, ylabel, title, path in [
    ("N_States", "Number of states",
     "Number of unique matched states per method", "out/n_states.png"),
    ("N_Segments_K", "Segments (\u00d710\u00b3)",
     "Number of segments per method", "out/n_segments.png"),
    ("Entropy", "Entropy (bits)",
     "Transition matrix entropy (full)", "out/entropy.png"),
    ("Entropy_NOQH", "Entropy (bits)",
     f"Transition matrix entropy ({NOQH_DISPLAY}, excl. Quies/Het)", "out/entropy_noqh.png"),
]:
    utils.bar_plot(df_results, "Method", column, order=method_order,
                   palette=method_palette, xticklabels="display",
                   title=title, ylabel=ylabel, labels="{:.2f}", path=path)

In [ ]:
# State composition and segment lengths per state and method.
figw = max(12, len(states_order) * len(method_order) * 0.22)

# Every state is present for every (Method, Dataset) pair, a state the pair
# never calls as a 0. observed=True: without it the categorical Method crosses
# with every Dataset, and each absent pair enters as an all-zero row.
df_comp_filled = (df_comp.pivot_table(index=["Method", "Dataset"], columns="State",
                                      values="Fraction", fill_value=0, observed=True)
                  .stack().reset_index(name="Fraction"))

utils.broken_bar_plot(df_comp_filled, "State", "Fraction", order=states_order,
                      hue="Method", hue_order=method_order, palette=method_palette,
                      break_low=BREAK_LOW, break_high=BREAK_HIGH, figsize=(figw, 6),
                      title="Average state composition per method",
                      xlabel="Chromatin state", ylabel="Fraction of genome",
                      legend=True, points=CROWDED_POINTS,
                      path="out/avg_composition.png")

for column, label in [("MeanLength", "mean"), ("MedianLength", "median")]:
    utils.bar_plot(df_comp, "State", column, order=states_order, hue="Method",
                   hue_order=method_order, palette=method_palette, figsize=(figw, 6),
                   title=f"Average state {label} length per method", xlabel="State",
                   ylabel=f"{label.capitalize()} length (bp, log scale)", log=True,
                   legend=True, points=CROWDED_POINTS,
                   path=f"out/avg_{label}_length.png")

# The same composition stacked: per method averaged over the epigenomes, and
# per epigenome for each method.
pivot_avg = (df_comp.pivot_table(index=["Method", "Dataset"], columns="State",
                                 values="Fraction", fill_value=0, observed=True)
             .groupby("Method", observed=True).mean().reindex(method_order))
pivot_avg = pivot_avg[[s for s in states_order if s in pivot_avg.columns]]
utils.stacked_bar_plot(pivot_avg, colors=[state_colors[s] for s in pivot_avg.columns],
                       figsize=(8, 5), width=0.6, rotation=45, tick_fontsize=8,
                       xticklabels=[utils.display_name(m) for m in method_order],
                       title="Average state composition per method",
                       xlabel="Method", ylabel="Average fraction of genome",
                       path="out/avg_composition_stacked.png")

for method in method_order:
    pivot = (df_comp[df_comp["Method"] == method]
             .pivot(index="Dataset", columns="State", values="Fraction").fillna(0))
    if pivot.empty:
        continue
    pivot = pivot[[s for s in states_order if s in pivot.columns]]
    utils.stacked_bar_plot(pivot, colors=[state_colors[s] for s in pivot.columns],
                           figsize=(20, 8), tick_fontsize=6 if len(pivot) > 50 else 8,
                           title=f"State composition per dataset - {utils.display_name(method)}",
                           xlabel="Dataset (EID)", ylabel="Fraction of genome",
                           path=f"out/composition_{utils.slug(method)}.png")

In [ ]:
# Pairwise consistency, pooled over the pairs and per state.
for mode, mode_key in DOMAINS:
    for metric, metric_key in METRICS:
        utils.bar_plot(df_pw[df_pw["Mode"] == mode], "Method", metric,
                       order=method_order, palette=method_palette, figsize=(6, 4.5),
                       xticklabels="display", labels="{:.2f}", points=PAIR_POINTS,
                       title=f"Pairwise {metric} consistency ({mode})",
                       ylabel=f"{metric} index",
                       path=f"out/pairwise_consistency_{mode_key}_{metric_key}.png")

for metric, metric_key in METRICS:
    utils.bar_plot(df_pw_state[df_pw_state["Metric"] == metric], "State", "Value",
                   order=states_order, hue="Method", hue_order=method_order,
                   palette=method_palette, figsize=(figw, 6), xlabel="State",
                   title=f"Average {metric} per state for all methods", ylabel=metric,
                   ylim=(0 if metric_key == JACCARD else None, 1.05), legend=True,
                   points={"size": 1, "alpha": 0.2, "jitter": 0.3},
                   path=f"out/pw_bar_all_{metric_key}.png")

### Show

In [ ]:
show_all(["n_peaks.png", "peak_length.png", "n_states.png", "n_segments.png",
          "avg_composition.png", "avg_mean_length.png", "avg_median_length.png",
          "avg_composition_stacked.png", "entropy.png", "entropy_noqh.png"], base="out")
show_all([f"pairwise_consistency_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")
for metric, metric_key in METRICS:
    show(f"out/pw_bar_all_{metric_key}.png",
         caption=f"{metric} of every state with itself, over the sampled pairs")
show_all([f"composition_{utils.slug(m)}.png" for m in method_order], base="out")

## 2. Reference 18-state core K27ac model

The published Roadmap 18-state segmentation of every epigenome: one model
trained over all of them, so every plot of this section carries the hatch of a
joint model.

### Compute

In [ ]:
core_18_paths = {os.path.basename(f).split("_")[0]: f for f in
                 sorted(glob.glob(f"{EPI_1000_PATH}/E*_18_core_K27ac_dense.bed.gz"))}
core_18_ids = sorted(core_18_paths)
print(f"{len(core_18_ids)} 18-state core K27ac segmentations")


def _stats_18(eid):
    return analyze.segmentation_stats(match.load_bed(core_18_paths[eid]), 200,
                                      background=NOQH_STATES)


_stats_18_cache = {}


def stats_18():
    """Stats of every 18-state segmentation - see analyze.segmentation_stats.

    Computed once and shared by the three caches below, so a single missing
    cache file does not walk the segmentations again.
    """
    if not _stats_18_cache:
        # Half the cores: a worker holds a whole segmentation, several hundred MB.
        with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() // 2),
                                 mp_context=mp.get_context("fork")) as executor:
            per_eid = dict(zip(core_18_ids,
                               tqdm(executor.map(_stats_18, core_18_ids),
                                    total=len(core_18_ids), desc="18-state stats")))
        _stats_18_cache.update(
            segments=pd.DataFrame([{"Dataset": eid, "N_Segments": s["n_segments"]}
                                   for eid, s in per_eid.items()]),
            comp=pd.DataFrame([{"Dataset": eid, **state}
                               for eid, s in per_eid.items() for state in s["composition"]]),
            entropy=pd.DataFrame([{"Dataset": eid, "Mode": utils.domain_display(mode),
                                   "Entropy": value}
                                  for eid, s in per_eid.items()
                                  for mode, value in s["entropy"].items()]),
            colors={state: color for s in per_eid.values()
                    for state, color in s["colors"].items()})
    return _stats_18_cache


df_segments_18 = utils.cached_csv("out/df_segments_18.csv", lambda: stats_18()["segments"],
                                  label="18-state segment counts")
df_comp_18 = utils.cached_csv("out/df_comp_18.csv", lambda: stats_18()["comp"],
                              label="18-state composition")
df_entropy_18 = utils.cached_csv("out/df_entropy_18.csv", lambda: stats_18()["entropy"],
                                 label="18-state entropy")
states_18 = summary_plots.sort_states(df_comp_18["State"].unique())
state_colors_18 = summary_plots.state_palette(
    states_18, utils.cached_json("out/state_colors_18.json", lambda: stats_18()["colors"],
                                 label="18-state colors"))


def compute_pw_18():
    """Metrics and overlaps of the sampled pairs of 18-state segmentations."""
    pairs = utils.sample_pairs([(id1, id2) for i, id1 in enumerate(core_18_ids)
                                for id2 in core_18_ids[i + 1:]], PAIR_LIMIT)
    left, rows, overlaps = {}, [], []
    for id1, id2 in tqdm(pairs, desc="Overlaps 18-core"):
        s1, l1 = load_side(left, core_18_paths[id1])
        s2 = match.load_bed(core_18_paths[id2])
        overlap = match.pair_overlap(s1, s2)
        overlaps.append(overlap)
        metrics = match.agreement_by_mode(overlap, l1, match.state_lengths(s2),
                                          background=NOQH_STATES)
        for mode, m in metrics.items():
            rows.append({"Dataset1": id1, "Dataset2": id2,
                         "Mode": utils.domain_display(mode),
                         JACCARD_DISPLAY: m[JACCARD], KAPPA_DISPLAY: m[KAPPA],
                         COSINE_DISPLAY: m[COSINE]})
    return {"df": pd.DataFrame(rows), "overlaps": overlaps}


cache_18 = utils.cached_pickle("out/pw_18_core_cache.pkl", compute_pw_18,
                               label="18-state pairwise metrics")
df_pw_18 = cache_18["df"]
df_pw_18.to_csv("out/df_pw_18.csv", index=False)
df_pw_state_18 = per_state_rows(cache_18["overlaps"], states_18)
del cache_18

### Plots

In [ ]:
# One model, so one bar per plot: the label is the model rather than a method.
LABEL_18 = "18-core"
bar_18 = dict(order=[LABEL_18], color="skyblue", hatch="all", figsize=(3, 4),
              rotation=0, label_fontsize=8)

utils.bar_plot(df_segments_18.assign(Model=LABEL_18), "Model", "N_Segments",
               title="Distribution of segment numbers (18-state core K27ac)",
               ylabel="Number of segments", labels="{:.0f}",
               path="out/epi_18_core_segments_dist.png", **bar_18)

for mode, mode_key in DOMAINS:
    utils.bar_plot(df_entropy_18[df_entropy_18["Mode"] == mode].assign(Model=LABEL_18),
                   "Model", "Entropy", labels="{:.3f}",
                   title=f"Transition Entropy ({mode})\n(18-state core K27ac)",
                   ylabel="Entropy (bits)",
                   path=f"out/epi_18_core_entropy_{mode_key}.png", **bar_18)
    for metric, metric_key in METRICS:
        utils.bar_plot(df_pw_18[df_pw_18["Mode"] == mode].assign(Model=LABEL_18),
                       "Model", metric, labels="{:.2f}", points=PAIR_POINTS,
                       title=f"Pairwise {metric} consistency ({mode}) - 18-core",
                       ylabel=f"{metric} index",
                       path=f"out/epi_18_core_pw_{mode_key}_{metric_key}.png", **bar_18)

# State composition and segment lengths, coloured by state.
df_comp_18_filled = (df_comp_18.pivot_table(index="Dataset", columns="State",
                                            values="Fraction", fill_value=0)
                     .stack().reset_index(name="Fraction"))
utils.broken_bar_plot(df_comp_18_filled, "State", "Fraction", order=states_18,
                      palette=state_colors_18, hatch=None,
                      break_low=BREAK_LOW, break_high=BREAK_HIGH, figsize=(10, 6),
                      title="Average state composition (18-state core K27ac)",
                      xlabel="State", ylabel="Average fraction of genome",
                      points=CROWDED_POINTS,
                      path="out/epi_18_core_avg_composition.png")

for column, label in [("MeanLength", "mean"), ("MedianLength", "median")]:
    utils.bar_plot(df_comp_18, "State", column, order=states_18,
                   palette=state_colors_18, hatch=None, figsize=(10, 5), log=True,
                   title=f"Average state {label} length (18-state core K27ac)",
                   xlabel="State", ylabel=f"{label.capitalize()} length (bp, log scale)",
                   path=f"out/epi_18_core_avg_{label}_length.png")

pivot_comp_18 = (df_comp_18.pivot(index="Dataset", columns="State", values="Fraction")
                 .fillna(0)[states_18])
utils.stacked_bar_plot(pivot_comp_18, colors=[state_colors_18[s] for s in states_18],
                       figsize=(15, 6),
                       title="State composition per dataset (18-state core K27ac)",
                       xlabel="Dataset", ylabel="Fraction of genome",
                       path="out/epi_18_core_composition_per_dataset.png")

for metric, metric_key in METRICS:
    utils.bar_plot(df_pw_state_18[df_pw_state_18["Metric"] == metric], "State", "Value",
                   order=states_18, palette=state_colors_18, hatch=None,
                   figsize=(10, 5), ylim=(0, 1.05), points={"size": 1, "alpha": 0.3, "jitter": 0.2},
                   title=f"18-state core Average {metric} per state",
                   xlabel="State", ylabel=metric,
                   path=f"out/pw_bar_18_core_{metric_key}.png")

### Show

In [ ]:
show_all(["epi_18_core_segments_dist.png", "epi_18_core_composition_per_dataset.png",
          "epi_18_core_avg_composition.png", "epi_18_core_avg_mean_length.png",
          "epi_18_core_avg_median_length.png"], base="out")
show_all([f"epi_18_core_entropy_{domain}.png" for _, domain in DOMAINS], base="out")
show_all([f"epi_18_core_pw_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")
for metric, metric_key in METRICS:
    show(f"out/pw_bar_18_core_{metric_key}.png",
         caption=f"{metric} of every state with itself, over the sampled pairs")

## 3. Reference 15-state core model: individual vs joint

The published 15-state core segmentations, called per epigenome (`chromhmm_default`)
against the joint model reordered onto the same states (`joint_chromhmm`), plus
the agreement of every sample with its own joint segmentation - for the de-novo
methods of section 1 as well.

### Compute

In [ ]:
indiv_15_paths = {os.path.basename(f).split("_")[0]: f for f in
                  sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_matched.bed"))}
joint_15_paths = {os.path.basename(f).split("_")[0]: f for f in
                  sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_joint_reodered.bed.gz")
                         + glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_joint_reordered.bed.gz"))}
ids_15 = sorted(set(indiv_15_paths) & set(joint_15_paths))

# The two published models, keyed the way every method of this notebook is; the
# display names only ever reach the plot labels.
REF_15_PATHS = {CHROMHMM_DEFAULT: indiv_15_paths, JOINT_CHROMHMM: joint_15_paths}
order_15 = list(REF_15_PATHS)
palette_15 = {CHROMHMM_DEFAULT: "lightcoral", JOINT_CHROMHMM: "skyblue"}
print(f"Found {len(ids_15)} common samples for the 15-state models")


def _stats_15(key):
    eid, method = key
    return analyze.segmentation_stats(match.load_bed(REF_15_PATHS[method][eid]), 200,
                                      background=NOQH_STATES)


stats_15 = utils.keyed_cache("out/stats_15_cache.pkl",
                             [(eid, method) for eid in ids_15 for method in order_15],
                             _stats_15, label="15-state segmentation stats",
                             valid=lambda stats: "entropy" in stats, progress=tqdm)

df_segments_15 = pd.DataFrame([{"Dataset": eid, "Method": method,
                                "N_Segments": stats["n_segments"]}
                               for (eid, method), stats in stats_15.items()])
df_comp_15 = pd.DataFrame([{"Dataset": eid, "Method": method, **state}
                           for (eid, method), stats in stats_15.items()
                           for state in stats["composition"]])
df_entropy_15 = pd.DataFrame([{"Dataset": eid, "Method": method,
                               "Mode": utils.domain_display(mode), "Entropy": value}
                              for (eid, method), stats in stats_15.items()
                              for mode, value in stats["entropy"].items()])
# Persisted next to df_pw_15.csv below: summary.ipynb reads the entropy and the
# segment counts of the two published 15-state models out of these, and without
# them on disk its 1000-epigenomes joint ChromHMM has no entropy,
# state-fidelity or segment-stability evidence.
df_segments_15.to_csv("out/df_segments_15.csv", index=False)
df_entropy_15.to_csv("out/df_entropy_15.csv", index=False)

states_15 = summary_plots.sort_states(df_comp_15["State"].unique())
state_colors_15 = summary_plots.state_palette(
    states_15, utils.cached_json(
        "out/state_colors_15.json",
        lambda: {state: color for stats in stats_15.values()
                 for state, color in stats["colors"].items()},
        label="15-state colors"))

In [ ]:
# Pairwise consistency of the two published models, over the same sampled pairs.
_left_15 = {}


def _pw_15(key):
    method, id1, id2 = key
    paths = REF_15_PATHS[method]
    s1, l1 = load_side(_left_15, paths[id1])
    s2 = match.load_bed(paths[id2])
    overlap = match.pair_overlap(s1, s2)
    metrics = match.agreement_by_mode(overlap, l1, match.state_lengths(s2),
                                      background=NOQH_STATES)
    # Metrics as the (jaccard, kappa, cosine) tuples out/pw_15_cache.pkl holds.
    return {"metrics": {mode: (m[JACCARD], m[KAPPA], m[COSINE])
                        for mode, m in metrics.items()},
            "overlap": overlap}


pairs_15 = utils.sample_pairs([(id1, id2) for i, id1 in enumerate(ids_15)
                               for id2 in ids_15[i + 1:]], PAIR_LIMIT)
pw_15 = utils.keyed_cache("out/pw_15_cache.pkl",
                          [(method, id1, id2) for method in order_15
                           for id1, id2 in pairs_15],
                          _pw_15, label="15-state pairwise metrics", progress=tqdm)

df_pw_15 = pd.DataFrame([{"Method": method, "Dataset1": id1, "Dataset2": id2,
                          "Mode": utils.domain_display(mode),
                          JACCARD_DISPLAY: jaccard, KAPPA_DISPLAY: kappa,
                          COSINE_DISPLAY: cosine}
                         for (method, id1, id2), value in pw_15.items()
                         for mode, (jaccard, kappa, cosine) in value["metrics"].items()])
df_pw_15.to_csv("out/df_pw_15.csv", index=False)

df_pw_state_15 = pd.concat(
    [per_state_rows([pw_15[(method, id1, id2)]["overlap"] for id1, id2 in pairs_15],
                    states_15, method=method) for method in order_15],
    ignore_index=True)
del pw_15

In [ ]:
# Individual vs joint: the same sample segmented on its own and inside a joint
# model. The reference 15-state pair is the published individual model against
# the reordered joint one; the de-novo pairs come from section 1.
JOINT_OF = {CHROMHMM_DEFAULT: JOINT_CHROMHMM, KMEANS_HOMER: JOINT_KMEANS_HOMER,
            KMEANS_MACS2: JOINT_KMEANS_MACS2, KMEANS_OMNI: JOINT_KMEANS_OMNI}

comp_tasks = [(CHROMHMM_DEFAULT, eid, indiv_15_paths[eid], joint_15_paths[eid])
              for eid in ids_15]
for method in (KMEANS_HOMER, KMEANS_MACS2, KMEANS_OMNI):
    joint = JOINT_OF[method]
    both = ({f for f, m in seg_by_key if m == method}
            & {f for f, m in seg_by_key if m == joint})
    comp_tasks += [(method, folder, seg_by_key[(folder, method)], seg_by_key[(folder, joint)])
                   for folder in sorted(both)]

joint_indiv_order = [m for m in JOINT_OF if any(t[0] == m for t in comp_tasks)]


def compute_joint_indiv():
    rows = []
    for method, eid, path_indiv, path_joint in tqdm(comp_tasks, desc="Individual vs joint"):
        metrics = pair_agreement(path_indiv, path_joint)
        for mode, mode_key in DOMAINS:
            rows.append({"Method": method, "Dataset": eid, "Mode": mode,
                         **{utils.metric_display(m): v
                            for m, v in metrics[mode_key].items()}})
    return pd.DataFrame(rows)


df_joint_indiv = utils.cached_csv(
    "out/df_joint_indiv_comparison.csv", compute_joint_indiv,
    label="Individual vs joint metrics",
    valid=lambda df: len(df.groupby(["Method", "Dataset"])) == len(comp_tasks)
                     and methods_current(df, joint_indiv_order))
print(f"{len(comp_tasks)} individual / joint pairs over "
      f"{len(joint_indiv_order)} methods")

### Plots

In [ ]:
# Segment numbers, composition, segment lengths and pairwise consistency of the
# two published 15-state models side by side.
bar_15 = dict(order=order_15, palette=palette_15, xticklabels="display",
              figsize=(4, 4), label_fontsize=8)

utils.bar_plot(df_segments_15, "Method", "N_Segments", labels="{:.0f}",
               title="Distribution of segment numbers (15-state)",
               ylabel="Number of segments",
               path="out/epi_15_segments_comparison.png", **bar_15)

for mode, mode_key in DOMAINS:
    utils.bar_plot(df_entropy_15[df_entropy_15["Mode"] == mode], "Method", "Entropy",
                   labels="{:.3f}", ylabel="Entropy (bits)",
                   title=f"Transition matrix entropy ({mode})\n(15-state models)",
                   path=f"out/epi_15_entropy_{mode_key}.png", **bar_15)
    for metric, metric_key in METRICS:
        utils.bar_plot(df_pw_15[df_pw_15["Mode"] == mode], "Method", metric,
                       labels="{:.2f}", points=PAIR_POINTS,
                       title=f"Pairwise {metric} consistency ({mode})\n(15-state models)",
                       ylabel=f"{metric} index",
                       path=f"out/epi_15_pw_{mode_key}_{metric_key}.png", **bar_15)

# The three metrics of the two published models in one figure, next to the
# per-metric plots above. "Composition" is the cosine of the state-composition
# vectors - the spelling compare.py writes that quantity under - so it is the
# Cosine column of df_pw_15 renamed, and it answers whether the two sides spend
# the genome on states in the same proportions; Kappa and Jaccard look at where
# they put them. FULL domain, so Quies/Het count towards all three.
df_pw_15_agreement = (df_pw_15[df_pw_15["Mode"] == FULL_DISPLAY]
                      .rename(columns={COSINE_DISPLAY: COMPOSITION_DISPLAY})
                      .melt(id_vars=["Method"],
                            value_vars=[COMPOSITION_DISPLAY, KAPPA_DISPLAY,
                                        JACCARD_DISPLAY],
                            var_name="Metric", value_name="Value"))
utils.bar_plot(df_pw_15_agreement, "Metric", "Value",
               order=[COMPOSITION_DISPLAY, KAPPA_DISPLAY, JACCARD_DISPLAY],
               hue="Method", hue_order=order_15, palette=palette_15,
               figsize=(6, 4.5), rotation=0, ylim=(0, 1.05), points=PAIR_POINTS,
               title="Pairwise agreement", xlabel="Metric",
               ylabel="Agreement index", legend=True, legend_title="Model",
               path="out/epi_15_pw_agreement.png")

df_comp_15_filled = (df_comp_15.pivot_table(index=["Method", "Dataset"], columns="State",
                                            values="Fraction", fill_value=0)
                     .stack().reset_index(name="Fraction"))
utils.broken_bar_plot(df_comp_15_filled, "State", "Fraction", order=states_15,
                      hue="Method", hue_order=order_15, palette=palette_15,
                      break_low=BREAK_LOW, break_high=BREAK_HIGH, figsize=(12, 6),
                      title="Average state composition comparison (15-state)",
                      xlabel="State", ylabel="Average fraction of genome",
                      legend=True, points=CROWDED_POINTS,
                      path="out/epi_15_avg_composition_comparison.png")

for column, label in [("MeanLength", "mean"), ("MedianLength", "median")]:
    utils.bar_plot(df_comp_15, "State", column, order=states_15, hue="Method",
                   hue_order=order_15, palette=palette_15, figsize=(12, 6), log=True,
                   title=f"Average state {label} length comparison (15-state)",
                   xlabel="State", ylabel=f"{label.capitalize()} length (bp, log scale)",
                   legend=True, points=CROWDED_POINTS,
                   path=f"out/epi_15_avg_{label}_length_comparison.png")

for metric, metric_key in METRICS:
    utils.bar_plot(df_pw_state_15[df_pw_state_15["Metric"] == metric], "State", "Value",
                   order=states_15, hue="Method", hue_order=order_15, palette=palette_15,
                   figsize=(12, 6), ylim=(0 if metric_key == JACCARD else None, 1.05),
                   title=f"15-state Average {metric} per state", xlabel="State",
                   ylabel=metric, legend=True, legend_title="Model",
                   points={"size": 1, "alpha": 0.2, "jitter": 0.3},
                   path=f"out/pw_bar_15_all_{metric_key}.png")

# The same composition stacked, averaged per model and per dataset.
colors_15 = [state_colors_15[s] for s in states_15]
pivot_avg_15 = (df_comp_15.pivot_table(index=["Method", "Dataset"], columns="State",
                                       values="Fraction", fill_value=0)
                .groupby("Method").mean().reindex(index=order_15, columns=states_15))
utils.stacked_bar_plot(pivot_avg_15, colors=colors_15, figsize=(8, 6), width=0.6,
                       rotation=0, tick_fontsize=8,
                       xticklabels=[utils.display_name(m) for m in order_15],
                       title="Average state composition comparison (15-state, stacked)",
                       xlabel="Method", ylabel="Average fraction of genome",
                       path="out/epi_15_avg_composition_stacked.png")

for method in order_15:
    pivot = (df_comp_15[df_comp_15["Method"] == method]
             .pivot(index="Dataset", columns="State", values="Fraction")
             .fillna(0).reindex(columns=states_15))
    utils.stacked_bar_plot(pivot, colors=colors_15, figsize=(15, 6),
                           title=f"State composition per dataset "
                                 f"({utils.display_name(method)} 15-state)",
                           xlabel="Dataset", ylabel="Fraction of genome",
                           path=f"out/epi_15_{utils.slug(method)}_composition_per_dataset.png")

In [ ]:
# Individual vs joint agreement, per method.
for mode, mode_key in DOMAINS:
    df_mode = df_joint_indiv[df_joint_indiv["Mode"] == mode]
    if df_mode.empty:
        continue
    for metric, metric_key in METRICS:
        utils.bar_plot(df_mode, "Method", metric, order=joint_indiv_order,
                       palette=method_palette, xticklabels="display", hatch=None,
                       ylim=(0, 1.05), labels="{:.2f}",
                       title=f"Individual vs joint ({mode}): {metric}", ylabel=metric,
                       path=f"out/joint_indiv_comparison_{mode_key}_{metric_key}.png")

# The reference 15-state pair on its own, its three metrics in one figure per
# domain: how much of a sample's individual segmentation survives being called
# inside the joint model. One point per epigenome, and the metric colours of
# analysis_encode.ipynb (summary_plots.SIMILARITY_COLORS), which is what the
# same three metrics are drawn in there. "Composition" is the cosine of the
# state-composition vectors, the Cosine column under compare.py's spelling.
#
# No hatch: a bar here is an individual model measured against a joint one, so
# neither side is the one the hatch would pick out.
df_joint_indiv_15 = (df_joint_indiv[df_joint_indiv["Method"] == CHROMHMM_DEFAULT]
                     .rename(columns={COSINE_DISPLAY: COMPOSITION_DISPLAY})
                     .melt(id_vars=["Dataset", "Mode"],
                           value_vars=summary_plots.SIMILARITY_ORDER,
                           var_name="Metric", value_name="Value"))
for mode, mode_key in DOMAINS:
    utils.bar_plot(df_joint_indiv_15[df_joint_indiv_15["Mode"] == mode],
                   "Metric", "Value", order=summary_plots.SIMILARITY_ORDER,
                   palette=summary_plots.SIMILARITY_COLORS, hatch=None,
                   figsize=(5, 4.5), rotation=0, ylim=(0, 1.05), labels="{:.2f}",
                   label_fontsize=8, points={"size": 2, "alpha": 0.5, "jitter": 0.2},
                   title=f"Individual vs Joint model ({mode})", xlabel="Metric",
                   ylabel="Agreement index",
                   path=f"out/epi_15_joint_indiv_{mode_key}.png")


In [ ]:
# Composition per method, the published reference next to every de-novo model.
# The reference of an epigenome is its joint 15-state core segmentation, the
# one the plots above compare the individual published model against; the
# de-novo side is section 1's, individual and joint models together.
#
# The two sides spell the same 15 states differently - the published model
# numbers them ("15_Quies"), the de-novo ones carry the bare names they were
# matched onto ("Quies") - so the number is stripped before they are pooled.
REF_LABEL = "Reference (joint 15-state)"
comp_order = [REF_LABEL] + [utils.display_name(m) for m in method_order]
comp_palette = {REF_LABEL: utils.bin_color("reference"),
                **{utils.display_name(m): utils.method_color(m) for m in method_order}}

df_comp_methods = pd.concat([
    df_comp_15[df_comp_15["Method"] == JOINT_CHROMHMM].assign(
        Model=REF_LABEL,
        State=lambda df: df["State"].str.replace(r"^\d+_", "", regex=True)),
    df_comp.assign(Model=lambda df: df["Method"].astype(str).map(utils.display_name)),
])[["Dataset", "Model", "State", "Fraction"]]

comp_states = summary_plots.sort_states(df_comp_methods["State"].unique())
comp_colors = summary_plots.state_palette(comp_states, state_colors, ref_colors)

# Every state of every model, a state a model never calls as a 0 rather than as
# a missing bar.
comp_pivot = df_comp_methods.pivot_table(index=["Model", "Dataset"], columns="State",
                                         values="Fraction", fill_value=0)

axes = utils.broken_bar_plot(
    comp_pivot.stack().reset_index(name="Fraction"), "State", "Fraction",
    order=comp_states, hue="Model", hue_order=comp_order, palette=comp_palette,
    break_low=BREAK_LOW, break_high=BREAK_HIGH,
    figsize=(max(12, len(comp_states) * len(comp_order) * 0.18), 6),
    title="State composition: reference and de-novo methods",
    xlabel="Chromatin state", ylabel="Fraction of genome",
    legend=True, legend_title="Model", points=CROWDED_POINTS)
# The reference is a joint model itself, and the default predicate of
# hatch_joint() does not read that off a label not starting with "joint", so
# the hatch of a joint model is put on its bars by name.
for ax in axes[0].figure.axes:
    utils.hatch_joint(ax, comp_order,
                      joint=lambda model: str(model).startswith(("Joint", REF_LABEL)))
utils.save_fig(axes[0].figure, "out/composition_methods.png", tight=False)

utils.stacked_bar_plot(
    comp_pivot.groupby("Model").mean().reindex(index=comp_order, columns=comp_states),
    colors=[comp_colors[s] for s in comp_states], figsize=(9, 5), width=0.6,
    rotation=45, tick_fontsize=8,
    title="State composition: reference and de-novo methods",
    xlabel="Model", ylabel="Average fraction of genome",
    path="out/composition_methods_stacked.png")


### Show

In [ ]:
show_all(["composition_methods.png", "composition_methods_stacked.png"],
         base="out")
show_all(["epi_15_segments_comparison.png", "epi_15_avg_composition_comparison.png",
          "epi_15_avg_composition_stacked.png", "epi_15_avg_mean_length_comparison.png",
          "epi_15_avg_median_length_comparison.png"]
         + [f"epi_15_{utils.slug(m)}_composition_per_dataset.png" for m in order_15],
         base="out")
show_all([f"epi_15_entropy_{domain}.png" for _, domain in DOMAINS], base="out")
show_all([f"epi_15_pw_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")
show("out/epi_15_pw_agreement.png")
for metric, metric_key in METRICS:
    show(f"out/pw_bar_15_all_{metric_key}.png",
         caption=f"{metric} of every state with itself, over the sampled pairs")
show_all([f"epi_15_joint_indiv_{domain}.png" for _, domain in DOMAINS], base="out")
show_all([f"joint_indiv_comparison_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")

## 4. Cell type differences in chromatin

How many of the epigenomes call the same state in the same place:
`analyze.compute_state_consistency` counts, per state, the number of
segmentations that agree on a bin (`w=0`) or within a window around it
(`w=1000`). A state whose mass sits at low support is cell-type specific, one
that peaks at full support is shared.

### Compute

In [ ]:
CONSISTENCY_WINDOWS = [1000, 0]

# The de-novo methods, the two published 15-state models and the 18-state one,
# each with the cache and figure names its counts are written under.
CONSISTENCY_MODELS = (
    [{"slug": f"denovo_{m}", "title": f"De-novo {utils.display_name(m)}",
      "png": f"out/epi_denovo_{m}",
      "paths": [seg_paths[i] for i in method_idxs[m]],
      "colors": None} for m in method_order]
    + [{"slug": f"15state_{m}", "title": f"15-state {utils.display_name(m)}",
        "png": f"out/epi_15_{m}",
        "paths": [REF_15_PATHS[m][eid] for eid in ids_15],
        "colors": state_colors_15} for m in order_15]
    + [{"slug": "18state_core", "title": "18-state Core", "png": "out/epi_18_core",
        "paths": [core_18_paths[eid] for eid in core_18_ids],
        "colors": state_colors_18}]
)

os.makedirs("out/consistency", exist_ok=True)


def consistency_counts(model, window):
    """{state: {support: bp}} of one model, cached under out/consistency/."""
    def compute():
        segs = [match.load_bed(p) for p in tqdm(model["paths"], desc="Loading BEDs",
                                                leave=False)]
        return analyze.compute_state_consistency(segs, window=window, show_progress=True)

    return utils.cached_pickle(f"out/consistency/{model['slug']}_w{window}.pkl", compute,
                               label=f"consistency for {model['title']} (w={window})")


consistency = {(model["slug"], window): consistency_counts(model, window)
               for model in CONSISTENCY_MODELS for window in CONSISTENCY_WINDOWS}

### Plots

In [ ]:
def n_segmentations(counts):
    """Number of segmentations behind a counts dict - its deepest support bucket.

    compute_state_consistency() keys each state on depth 1..M, so this recovers
    M without the file lists (plot_state_consistency does the same internally).
    """
    return max(depth for depths in counts.values() for depth in depths)


for model in CONSISTENCY_MODELS:
    print(f"--- {model['title']} ---")
    full_support = consistency[(model["slug"], 0)]
    depth = n_segmentations(full_support)
    for state in sorted(full_support, key=analyze._natural_sort_key):
        print(f"  {state}: {full_support[state][depth]}")
    # The colours the segmentations themselves carry, for a de-novo model.
    colors = model["colors"] or match.state_colors(match.load_bed(model["paths"][0]))
    for window in CONSISTENCY_WINDOWS:
        analyze.plot_state_consistency(consistency[(model["slug"], window)],
                                       f"{model['title']} (w={window})",
                                       f"{model['png']}_w{window}_consistency.png",
                                       colors=colors)

### Show

In [ ]:
show_all([f"{model['png']}_w{window}_consistency.png"
          for model in CONSISTENCY_MODELS for window in CONSISTENCY_WINDOWS])

## 5. Replicates consistency

Agreement between segmentations of samples that are replicates of the same
biological condition - the groups of `epi1000_replicates.yaml`, which differ
only in donor, cell line, sex or consortium. Unlike the pairwise consistency of
the sections above, computed over arbitrary sample pairs, this measures
reproducibility: how much of the disagreement between two segmentations comes
from the method rather than from real biology.

Jaccard, Kappa and Cosine, full and excluding the Quies/Het background, for
every individual and joint model, including the published 15-state reference in
both variants, plus the emission similarity of the state signatures the models
learn from each other's replicates.

### Compute

In [ ]:
# Replicate groups of the 127-epigenome set: samples of the same biological
# condition that differ only in donor / cell line / lab. Agreement within a
# group is the reproducibility ceiling of a segmentation method.
with open(os.path.join(PROJECT_ROOT, "epi1000_replicates.yaml")) as f:
    replicate_groups = yaml.safe_load(f)["epi1000replicates"]

# Every model of this notebook: the de-novo ones keyed by (sample, method) from
# valid_tasks, plus the published 15-state reference in both variants. Those
# two keep their own labels, which is how summary.ipynb tells the published
# models of this cache apart from the de-novo ones.
REF_15_LABELS = {CHROMHMM_DEFAULT: "Ref 15", JOINT_CHROMHMM: "Joint Ref 15"}
repl_paths = dict(seg_by_key)
for method, label in REF_15_LABELS.items():
    for eid, path in REF_15_PATHS[method].items():
        repl_paths[(eid, label)] = path

repl_method_order = method_order + list(REF_15_LABELS.values())

# Only groups with at least two samples segmented by the method contribute
# pairs; 29 of the 127 epigenomes are absent from this run, which empties some.
repl_tasks = [(method, group, eid1, eid2, repl_paths[(eid1, method)], repl_paths[(eid2, method)])
              for group, info in replicate_groups.items()
              for method in repl_method_order
              for eid1, eid2 in combinations([e for e in info["eids"]
                                              if (e, method) in repl_paths], 2)]

repl_group_order = [g for g in replicate_groups if any(t[1] == g for t in repl_tasks)]
repl_method_order = [m for m in repl_method_order if any(t[0] == m for t in repl_tasks)]
print(f"Replicate groups with pairs: {len(repl_group_order)} of {len(replicate_groups)}")
for group in repl_group_order:
    eids = sorted({e for t in repl_tasks if t[1] == group for e in t[2:4]})
    print(f"  {group:22s} {', '.join(eids)} ({replicate_groups[group]['varies']})")
print(f"{len(repl_tasks)} replicate pairs over {len(repl_method_order)} models")


def _replicate_pair(task):
    """The metrics of one replicate pair, one row per comparison domain."""
    method, group, eid1, eid2, path1, path2 = task
    metrics = pair_agreement(path1, path2)
    return [{"Method": method, "Group": group, "EID1": eid1, "EID2": eid2,
             "Mode": utils.domain_display(mode),
             JACCARD_DISPLAY: m[JACCARD], KAPPA_DISPLAY: m[KAPPA],
             COSINE_DISPLAY: m[COSINE]}
            for mode, m in metrics.items()]


def compute_replicates():
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as executor:
        rows = list(tqdm(executor.map(_replicate_pair, repl_tasks), total=len(repl_tasks)))
    return pd.DataFrame([row for pair_rows in rows for row in pair_rows])


df_repl = utils.cached_csv(
    "out/df_replicates.csv", compute_replicates, label="replicate consistency metrics",
    valid=lambda df: len(df.groupby(["Method", "EID1", "EID2"])) == len(repl_tasks)
                     and methods_current(df, repl_method_order))
df_repl["Method"] = pd.Categorical(df_repl["Method"], categories=repl_method_order, ordered=True)
df_repl = df_repl.sort_values(["Method", "Group", "Mode"]).reset_index(drop=True)

In [ ]:
# Replicate emission similarity: do the two segmentations of a condition
# describe it with the same state signatures, and not only in the same places?
#
# The emission matrix of a segmentation is the mean binarized signal of every
# mark over the bins of every state, so it needs the binarization behind the
# segmentation. chromhmm_binary/ keeps it for the ChromHMM and the reference
# models (200 bp bins, written by BinarizeBed); nothing here keeps the KMeans
# one, so it is rebuilt from the peak files at the KMEANS_BIN resolution
# process_epi1000.sh segmented them at. A caller's binarization is shared by
# its individual and its joint model, so each one is built once.
EMISSION_MARKS = ["H3K4me3", "H3K4me1", "H3K36me3", "H3K9me3", "H3K27me3", "H3K27ac"]
CHROMSIZES = os.path.join(EPI_1000_PATH, "hg19.chrom.sizes")

# Peak files of one sample and mark, the globs process_epi1000.sh binarized.
PEAK_GLOBS = {HOMER: "{eid}/homer/*{mark}*_homer.bed",
              MACS2: "{eid}/macs2/*{mark}*Peak",
              OMNI:  "{eid}/omni/*{mark}*.peak"}

# Binarization each model's emissions are read from: its own peak caller for a
# KMeans model, the ChromHMM one (None) for everything else.
EMISSION_SOURCE = {
    CHROMHMM_DEFAULT:               None,
    JOINT_CHROMHMM:                 None,
    REF_15_LABELS[CHROMHMM_DEFAULT]: None,
    REF_15_LABELS[JOINT_CHROMHMM]:   None,
    KMEANS_HOMER:       HOMER,
    KMEANS_MACS2:       MACS2,
    KMEANS_OMNI:        OMNI,
    JOINT_KMEANS_HOMER: HOMER,
    JOINT_KMEANS_MACS2: MACS2,
    JOINT_KMEANS_OMNI:  OMNI,
}

# Only the samples a replicate pair is built from need emissions.
repl_eids = sorted({e for t in repl_tasks for e in t[2:4]})
emission_tasks = []
for eid in repl_eids:
    by_source = defaultdict(list)
    for method in repl_method_order:
        path = repl_paths.get((eid, method))
        if path is not None:
            by_source[EMISSION_SOURCE[method]].append((method, path))
    emission_tasks += [(eid, caller, models) for caller, models in by_source.items()]

emission_keys = {(eid, method) for eid, _, models in emission_tasks for method, _ in models}
print(f"{len(emission_keys)} emission matrices over {len(repl_eids)} replicate "
      f"samples, from {len(emission_tasks)} binarizations")


def _emissions(task):
    """Emissions of every model of one sample that shares one binarization.

    Returns the matrices keyed by (sample, model) - None for a model whose
    binarization came out empty - and the marks the caller returned no peak
    for, which are a flat zero column of the emission matrix.
    """
    eid, caller, models = task
    blank = []
    if caller is None:
        by_chrom, marks, bin_size = {}, None, CHROMHMM_BIN
        for path in sorted(glob.glob(f"{eid}/chromhmm_binary/*_binary.txt*")):
            chrom, file_marks, data = analyze.load_binary(path)
            if len(data) == 0:
                continue
            marks = marks or file_marks
            by_chrom[chrom] = data
    else:
        peaks = [sorted(glob.glob(PEAK_GLOBS[caller].format(eid=eid, mark=mark)))
                 for mark in EMISSION_MARKS]
        chroms, sizes = peaks_segmentation.read_chrom_sizes(CHROMSIZES)
        # binarize_peaks reports every file it reads on stderr, which would
        # bury the progress bar under one line per mark per sample.
        with contextlib.redirect_stderr(io.StringIO()):
            binarized, per_chrom = peaks_segmentation.binarize_peaks(
                peaks, chroms, sizes, KMEANS_BIN, EMISSION_MARKS)
        # A caller returning no peak for a mark is an outcome, not a failure -
        # HOMER does it wherever it finds no enrichment - so the mark is kept
        # as the zero column it is. Dropping it instead would compare the two
        # sides of a pair over different marks, and rescale the cosine of every
        # state of a sample by whichever marks its caller happened to call.
        blank = [mark for mark, seen in zip(EMISSION_MARKS, binarized.max(axis=0))
                 if not seen]
        by_chrom, marks, bin_size = dict(per_chrom), EMISSION_MARKS, KMEANS_BIN

    out = {}
    for method, path in models:
        if not by_chrom or not marks:
            out[(eid, method)] = None
            continue
        out[(eid, method)] = analyze.state_emissions(
            match.load_bed(path), by_chrom, marks, bin_size)
    return out, blank


def compute_emissions_matrices():
    """One emission matrix per replicate sample and model."""
    matrices, blank = {}, []
    # Half the cores: a worker holds the whole binarized genome, ~200 MB.
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count() // 2),
                             mp_context=mp.get_context("fork")) as executor:
        for (out, task_blank), (eid, caller, _) in zip(
                tqdm(executor.map(_emissions, emission_tasks), total=len(emission_tasks)),
                emission_tasks):
            matrices.update(out)
            if task_blank:
                blank.append((eid, caller, task_blank))
    if blank:
        print(f"  {sum(len(marks) for _, _, marks in blank)} marks of "
              f"{len(blank)} sample / caller combinations have no peaks, and "
              f"stay a zero column of the emissions:")
        for eid, caller, marks in blank:
            print(f"    {eid} {caller}: {', '.join(marks)}")
    return matrices


repl_emissions = utils.cached_pickle(
    "out/epi_repl_emissions.pkl", compute_emissions_matrices,
    label="replicate emission matrices", valid=lambda m: set(m) == emission_keys)


def emission_cosine(a, b):
    """Mean cosine of the emission vectors of two segmentations' states, over
    the one-to-one state matching that maximizes it."""
    if a is None or b is None:
        return np.nan
    states_a, marks_a, mat_a = a
    states_b, marks_b, mat_b = b
    # Both sides come from the same binarization, so they carry the same marks;
    # they are still lined up by name rather than by position, since a cosine
    # over misaligned marks would be quietly wrong instead of missing.
    marks = [m for m in marks_a if m in marks_b]
    if not marks:
        return np.nan
    similarity, _ = match.emission_cosine_mapping(
        states_a, mat_a[:, [marks_a.index(m) for m in marks]],
        states_b, mat_b[:, [marks_b.index(m) for m in marks]])
    return similarity


# Cheap next to the matrices above - a 15x15 assignment problem per pair - so
# it is recomputed rather than cached, and only written out for summary.ipynb.
df_repl_em = pd.DataFrame(
    [{"Method": method, "Group": group, "EID1": eid1, "EID2": eid2,
      EMISSION_DISPLAY: emission_cosine(repl_emissions.get((eid1, method)),
                                        repl_emissions.get((eid2, method)))}
     for method, group, eid1, eid2, _, _ in repl_tasks]).dropna(subset=[EMISSION_DISPLAY])
df_repl_em["Method"] = pd.Categorical(df_repl_em["Method"],
                                      categories=repl_method_order, ordered=True)
df_repl_em = df_repl_em.sort_values(["Method", "Group"]).reset_index(drop=True)
df_repl_em.to_csv("out/df_replicates_emissions.csv", index=False)

print(f"\n{len(df_repl_em)} of {len(repl_tasks)} replicate pairs have both "
      f"emission matrices, mean {EMISSION_DISPLAY.lower()} per model:")
print(df_repl_em.groupby("Method", observed=True)[EMISSION_DISPLAY]
      .agg(["mean", "std", "size"]).round(3).to_string())

### Plots

In [ ]:
# The published reference models keep the colours of the 15-state plots above.
repl_palette = dict(method_palette, **{REF_15_LABELS[CHROMHMM_DEFAULT]: "lightcoral",
                                       REF_15_LABELS[JOINT_CHROMHMM]: "skyblue"})
# A group of two samples contributes a single pair, which the labels spell out.
group_labels = []
for group in repl_group_order:
    n_pairs = len({(t[2], t[3]) for t in repl_tasks if t[1] == group})
    group_labels.append(f"{group}\n({n_pairs} pair{'s' if n_pairs > 1 else ''})")

pooled = dict(order=repl_method_order, palette=repl_palette, xticklabels="display",
              figsize=(7, 4.5), labels="{:.2f}", points={"size": 2.5, "alpha": 0.6})
per_group = dict(order=repl_group_order, hue="Method", hue_order=repl_method_order,
                 palette=repl_palette, xticklabels=group_labels,
                 figsize=(max(10, len(repl_group_order) * 1.6), 4.5),
                 points={"size": 1.5, "alpha": 0.5, "jitter": 0.15}, legend=True,
                 legend_title="Model", legend_kwargs={"ncol": 2, "fontsize": 7,
                                                      "title_fontsize": 8},
                 err_kws={"linewidth": 1.2})

for mode, mode_key in DOMAINS:
    df_mode = df_repl[df_repl["Mode"] == mode]
    for metric, metric_key in METRICS:
        # Pooled over the groups, and per group: which conditions reproduce,
        # and where the models disagree about it.
        utils.bar_plot(df_mode, "Method", metric,
                       title=f"Replicate {metric} consistency ({mode})",
                       ylabel=f"{metric} index",
                       path=f"out/epi_repl_{mode_key}_{metric_key}.png", **pooled)
        utils.bar_plot(df_mode, "Group", metric,
                       title=f"Replicate {metric} consistency per group ({mode})",
                       ylabel=f"{metric} index",
                       path=f"out/epi_repl_group_{mode_key}_{metric_key}.png", **per_group)

# The two published 15-state models on their own: the replicate counterpart of
# the pairwise plot of section 3, where the de-novo bars of the plots above
# would otherwise set the scale of a two-bar comparison.
_repl_15 = df_repl[(df_repl["Mode"] == FULL_DISPLAY)
                   & df_repl["Method"].isin(REF_15_LABELS.values())]
utils.bar_plot(_repl_15, "Method", KAPPA_DISPLAY,
               order=[REF_15_LABELS[m] for m in order_15],
               xticklabels=[utils.display_name(m) for m in order_15],
               palette=repl_palette, figsize=(4, 4), label_fontsize=8,
               labels="{:.2f}", points={"size": 2.5, "alpha": 0.6},
               title=f"Replicate {KAPPA_DISPLAY} consistency ({FULL_DISPLAY})"
                     f"\n(15-state models)",
               ylabel=f"{KAPPA_DISPLAY} index",
               path=f"out/epi_repl_15_{FULL}_{KAPPA}.png")

# The three metrics of the same two models in one figure, the replicate
# counterpart of section 3's "Pairwise agreement" plot: the same bars measured
# over replicate pairs of one condition rather than over arbitrary sample
# pairs. "Composition" is the cosine of the state-composition vectors - the
# spelling compare.py writes that quantity under - so it is the Cosine column
# renamed, and it answers whether the two replicates spend the genome on states
# in the same proportions; Kappa and Jaccard look at where they put them.
# FULL domain, so Quies/Het count towards all three.
_repl_15_keys = {label: method for method, label in REF_15_LABELS.items()}
df_repl_15_agreement = (_repl_15
                        .assign(Method=_repl_15["Method"].astype(str).map(_repl_15_keys))
                        .rename(columns={COSINE_DISPLAY: COMPOSITION_DISPLAY})
                        .melt(id_vars=["Method", "Group"],
                              value_vars=[COMPOSITION_DISPLAY, KAPPA_DISPLAY,
                                          JACCARD_DISPLAY],
                              var_name="Metric", value_name="Value"))
utils.bar_plot(df_repl_15_agreement, "Metric", "Value",
               order=[COMPOSITION_DISPLAY, KAPPA_DISPLAY, JACCARD_DISPLAY],
               hue="Method", hue_order=order_15, palette=palette_15,
               figsize=(6, 4.5), rotation=0, ylim=(0, 1.05),
               points={"size": 2.5, "alpha": 0.6, "jitter": 0.15},
               title="Replicate agreement", xlabel="Metric",
               ylabel="Agreement index", legend=True, legend_title="Model",
               path="out/epi_15_repl_agreement.png")

# Emission similarity has one value per pair and no comparison domain: the
# emissions of the background states are part of a state signature like any
# other.
emission_ylabel = "Mean cosine of the matched state emissions"
utils.bar_plot(df_repl_em, "Method", EMISSION_DISPLAY,
               title="Replicate emission similarity", ylabel=emission_ylabel,
               path="out/epi_repl_emissions.png", **pooled)
utils.bar_plot(df_repl_em, "Group", EMISSION_DISPLAY,
               title="Replicate emission similarity per group", ylabel=emission_ylabel,
               path="out/epi_repl_group_emissions.png", **per_group)

### Show

In [ ]:
show_all([f"epi_repl_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")
show_all([f"epi_repl_group_{domain}_{metric}.png"
          for _, domain in DOMAINS for _, metric in METRICS], base="out")
show(f"out/epi_repl_15_{FULL}_{KAPPA}.png")
show("out/epi_15_repl_agreement.png")
show_all(["epi_repl_emissions.png", "epi_repl_group_emissions.png"], base="out")